# KALIA — Training (GPU T4 x2)

**Notebook settings**
- Accelerator: **GPU T4 x2**
- Internet: **On**
- Add-ons → Secrets: add `HF_TOKEN` (a HuggingFace *write* token)
- Add the **kalia-tokens** dataset to this notebook

Checkpoints sync to your private HF repo every 30 minutes, so sessions can be interrupted safely. Re-running the last cell resumes exactly where it stopped.

In [ ]:
!pip install -q tiktoken pyyaml huggingface_hub

In [ ]:
GITHUB_REPO = "https://github.com/CHANGE_ME/kalia.git"  # <-- edit this

!git clone {GITHUB_REPO} || (cd kalia && git pull)
%cd kalia

In [ ]:
import glob
import os
from pathlib import Path

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

hits = glob.glob("/kaggle/input/**/train.bin", recursive=True)
assert hits, "Attach the kalia-tokens dataset to this notebook"
DATA_DIR = str(Path(hits[0]).parent)
out_dir = "/kaggle/working/out"
HUB_REPO = "CHANGE_ME/kalia-m"  # <-- your HF repo: (org or user)/name
print("DATA_DIR =", DATA_DIR)

In [ ]:
%cd /kaggle/working/kalia
!torchrun --nproc_per_node=2 --standalone train.py --config configs/kalia-m.yaml --data-dir {DATA_DIR} --out-dir {out_dir} --resume --hub-repo {HUB_REPO} --max-minutes 510

**Notes**
- If torchrun complains about 2 GPUs, change `--nproc_per_node=2` to `1`.
- First run: no checkpoint exists yet, so it starts from random weights.
- Later sessions: automatically resumes from the latest checkpoint on HF Hub.
- Progress lives in your HF repo at `logs/train_log.csv`.